# Gatekeeper, giai đoạn 0 (§37): Kaggle có chấm được task SWE-smith với cùng điểm không?

**Trước khi chạy** (khung Settings bên phải):
- **Internet: On** (cần xác minh số điện thoại);
- **Accelerator: None** (không cần GPU);
- Persistence: không cần.

Chạy lần lượt ô 1–4 (ô 5 cũ đã chuyển sang notebook Benchmarks, xem `kaggle/run_stage0.md`). Ô 4 mất khoảng 30–60 phút và **chạy tiếp được** nếu bị ngắt (chạy lại ô đó).
Không ô nào in khóa hay thông tin tài khoản. Kết quả nằm ở `/kaggle/working/`: tải ba file `k0*.json*` về và gửi lại.

In [ ]:
# Ô 1 — cài đặt (khoảng 2–3 phút)
!pip -q install "swesmith==0.0.9" "swebench==4.1.0" udocker unidiff 2>&1 | tail -2
!udocker --allow-root install 2>&1 | tail -1
!docker info > /dev/null 2>&1 && echo "Docker daemon: CO" || echo "Docker daemon: KHONG"
!df -h /kaggle/working | tail -1

In [ ]:
# Ô 2 — mã và dữ liệu (repo công khai; không cần đăng nhập)
import os
import subprocess

os.chdir("/kaggle/working")
REPO = "dinhthithuthuyduphong-bot/gatekeeper"
if not os.path.isdir("gatekeeper"):
    url = f"https://github.com/{REPO}"
    branch = "claude/gatekeeper-13-research-nxytoh"  # Kaggle work lives here until it runs
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", branch, url], check=True)
RAW = f"https://raw.githubusercontent.com/{REPO}"
files = {
    "swesmith_pallets__jinja.jsonl": "data-snapshots/data/instances/swesmith_pallets__jinja.jsonl",
    "f1_probe.json": "trial/results/f1_probe.json",
    "f1_skill_probe.json": "trial/results/f1_skill_probe.json",
}
os.makedirs("k0data", exist_ok=True)
for name, path in files.items():
    dest = f"k0data/{name}"
    if not os.path.exists(dest):
        subprocess.run(["curl", "-sSfL", "-o", dest, f"{RAW}/{path}"], check=True)
    print(name, os.path.getsize(dest), "byte")
head = subprocess.run(
    ["git", "-C", "gatekeeper", "log", "--oneline", "-1"], capture_output=True, text=True
)
print(head.stdout)
# where the stage-0 code lives: kaggle/scripts on the default branch, scripts/ once moved to main
CODE = next(
    d
    for d in ("gatekeeper/kaggle/scripts", "gatekeeper/scripts")
    if os.path.exists(f"/kaggle/working/{d}/udocker_shim.py")
)
CODE = f"/kaggle/working/{CODE}"
print("code:", CODE)

In [ ]:
# Ô 3 — K0a: image của task có chạy được không (khoảng 2–3 phút cho lần đầu)
import json
import subprocess
import sys

sys.path.insert(0, CODE)
import udocker_shim  # noqa: E402

img = "swebench/swesmith.x86_64.pallets_1776_jinja.ada0a9a6"
name = udocker_shim.base_container(img)
inner = "git log --oneline | head -2; python --version; "
inner += "python -m pytest -q tests/test_api.py 2>&1 | tail -2"
cmd = ["udocker", "--allow-root", "--quiet", "run", "--nobanner", "--workdir=/testbed", name]
r = subprocess.run([*cmd, "/bin/bash", "-lc", inner], capture_output=True, text=True)
print(r.stdout[-600:], r.stderr[-300:])
with open("/kaggle/working/k0a.json", "w") as fh:
    json.dump({"returncode": r.returncode, "tail": r.stdout[-600:]}, fh)

In [ ]:
# Ô 4 — K0b: chấm lại các bản vá đã lưu, so với điểm đã ghi dưới Docker
# (30–60 phút; chạy lại được: các lượt đã chấm được bỏ qua)
import os
import shutil
import subprocess
import sys

os.makedirs("/kaggle/working/k0run", exist_ok=True)
os.chdir("/kaggle/working/k0run")  # SWE-smith writes its logs under ./logs/run_evaluation
D = "/kaggle/working/k0data"
cmd = [
    sys.executable,
    f"{CODE}/k0_stage0.py",
    "--rows",
    f"{D}/swesmith_pallets__jinja.jsonl",
    "--results",
    f"{D}/f1_probe.json",
    f"{D}/f1_skill_probe.json",
    "--controls",
    "20",
    "--out",
    "/kaggle/working/k0b_kaggle.jsonl",
]
p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:  # live progress, one line per grading
    print(line, end="")
print("exit code:", p.wait())
shutil.rmtree("/kaggle/working/k0run/logs", ignore_errors=True)

## Sau khi chạy xong ô 4

Gửi lại file `/kaggle/working/k0a.json` và `/kaggle/working/k0b_kaggle.jsonl` (khung Output, nút Download).

K0c và K0d (gọi model bằng hạn mức AI) **không** chạy được ở notebook thường: thư viện `kaggle_benchmarks` chỉ có sẵn trong notebook tạo từ https://www.kaggle.com/benchmarks/tasks/new. Xem `kaggle/notebooks/k0cd_benchmarks_cell.py` và `kaggle/run_stage0.md`.